# CropCare RL — Crop Disease Management Agent
**ALU Deep Learning Summative** | Cedric

This notebook trains four RL algorithms (DQN, PPO, A2C, REINFORCE) on a custom crop disease management environment and produces all results required for the report.


In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import subprocess, sys, os

# Clone repo (replace with your actual GitHub URL after pushing)
# subprocess.run(['git', 'clone', 'https://github.com/YOUR_USER/cedric_rl_summative.git'])
# os.chdir('cedric_rl_summative')

# If running from uploaded zip, extract here
# import zipfile
# with zipfile.ZipFile('/kaggle/input/YOUR_DATASET/cedric_rl_summative.zip') as z:
#     z.extractall('.')
# os.chdir('cedric_rl_summative')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])
print('Setup complete ✓')

In [ ]:
# ── Cell 2: Environment sanity check ───────────────────────────────────────
import os, sys
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

sys.path.insert(0, '.')
from environment.custom_env import CropDiseaseEnv, ACTION_LABELS

env = CropDiseaseEnv()
obs, info = env.reset(seed=0)
print(f'Observation space : {env.observation_space}')
print(f'Action space      : {env.action_space}')
print(f'Actions           : {list(ACTION_LABELS.values())}')
print(f'Diseased cells    : {info["total_diseased"]} / {env.grid_size**2}')
print(f'Obs shape         : {obs.shape}')
env.close()

In [ ]:
# ── Cell 3: Random agent demo (saves video) ─────────────────────────────────
!python random_demo.py --episodes 1 --steps 150

In [ ]:
# ── Cell 4: Train DQN — all 10 configs ─────────────────────────────────────
# For quick run use --timesteps 50000, full run use 200000
!python -m training.dqn_training --all --timesteps 100000 --verbose 0

In [ ]:
# ── Cell 5: Train PPO — all 10 configs ─────────────────────────────────────
!python -m training.pg_training --algo ppo --all --timesteps 100000 --verbose 0

In [ ]:
# ── Cell 6: Train A2C — all 10 configs ─────────────────────────────────────
!python -m training.pg_training --algo a2c --all --timesteps 100000 --verbose 0

In [ ]:
# ── Cell 7: Train REINFORCE — all 10 configs ────────────────────────────────
!python -m training.reinforce --all --episodes 1500

In [ ]:
# ── Cell 8: View results tables ─────────────────────────────────────────────
import pandas as pd

for algo in ['dqn', 'ppo', 'a2c', 'reinforce']:
    path = f'results/{algo}_results.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'\n=== {algo.upper()} Results ===')
        display(df.sort_values('mean_ep_reward', ascending=False))
        print(f'Best run: {df.loc[df.mean_ep_reward.idxmax(), "run"]} — '
              f'mean_reward={df.mean_ep_reward.max():.2f}')

In [ ]:
# ── Cell 9: Display all training plots ──────────────────────────────────────
import glob
from IPython.display import Image, display

plot_files = sorted(glob.glob('results/plots/*.png'))
print(f'Found {len(plot_files)} plots:')
for p in plot_files:
    print(f'  {p}')
    display(Image(p))

In [ ]:
# ── Cell 10: Run best agent simulation (with recording) ─────────────────────
# Find best algorithm by comparing CSVs
import pandas as pd, os

best_algo, best_run, best_score = None, None, -1e9
for algo in ['dqn', 'ppo', 'a2c', 'reinforce']:
    path = f'results/{algo}_results.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        idx = df.mean_ep_reward.idxmax()
        score = df.loc[idx, 'mean_ep_reward']
        if score > best_score:
            best_score = score
            best_algo  = algo
            best_run   = int(df.loc[idx, 'run'])

print(f'Best algorithm: {best_algo.upper()} run {best_run} — score {best_score:.2f}')
os.system(f'python main.py --algo {best_algo} --run {best_run} --episodes 5 --record')

In [ ]:
# ── Cell 11: Generate convergence comparison plot ───────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('All Algorithms — Hyperparameter Experiment Summary',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

algo_colors = {'dqn': 'steelblue', 'ppo': 'teal',
               'a2c': 'darkorange', 'reinforce': 'darkred'}

for i, algo in enumerate(['dqn', 'ppo', 'a2c', 'reinforce']):
    path = f'results/{algo}_results.csv'
    if not os.path.exists(path):
        continue
    df   = pd.read_csv(path).sort_values('run')
    ax   = axes[i]
    col  = algo_colors[algo]
    bars = ax.bar(df['run'], df['mean_ep_reward'],
                  yerr=df['std_ep_reward'], capsize=4,
                  color=col, edgecolor='black', linewidth=0.6, alpha=0.85)
    ax.set_title(f'{algo.upper()} — Mean Episode Reward per Config',
                 fontweight='bold')
    ax.set_xlabel('Run ID');  ax.set_ylabel('Mean Reward')
    ax.set_xticks(df['run'])
    best_run = df.loc[df.mean_ep_reward.idxmax(), 'run']
    ax.axvline(best_run, color='gold', linewidth=2, linestyle='--',
               label=f'Best: run {int(best_run)}')
    ax.legend(fontsize=9)

fig.tight_layout()
path = 'results/plots/all_algorithms_comparison.png'
fig.savefig(path, dpi=130, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')
from IPython.display import Image
Image(path)